# Crop Health Model Training (PyTorch) - Normal vs Infected
Run on **Google Colab** with GPU (Runtime → Change runtime type → T4 GPU).

`dataset.zip` must contain two class folders: `normal/` and `infected/` (a single top-level folder wrapping them is fine).
The notebook fine-tunes a pretrained MobileNetV2 (fast and accurate) and exports a TorchScript `model.pt` + `labels.json` for the Raspberry Pi.

In [ ]:
# Colab already has torch/torchvision/numpy/scikit-learn; this just makes sure they're present
!pip install -q torch torchvision numpy scikit-learn

## 1. Upload dataset.zip

In [ ]:
from google.colab import files
import os, shutil, zipfile

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

shutil.rmtree('dataset_raw', ignore_errors=True)
uploaded = files.upload()   # choose dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('dataset_raw')

def count_imgs(path):
    return sum(f.lower().endswith(IMG_EXT) for _, _, fs in os.walk(path) for f in fs)

# show what the zip actually contains
for root, dirs, fs in os.walk('dataset_raw'):
    if '__MACOSX' in root: continue
    depth = root.count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/', f"({count_imgs(root)} images)" if depth < 3 else '')

# find the folder whose subfolders are the classes (>=2 subfolders that contain images)
DATA_DIR = None
for root, dirs, _ in os.walk('dataset_raw'):
    if '__MACOSX' in root: continue
    classes = [d for d in dirs if d != '__MACOSX' and count_imgs(os.path.join(root, d)) > 0]
    if len(classes) >= 2:
        DATA_DIR = root
        break
assert DATA_DIR, "No folder with 2+ class subfolders of images found - check the structure printed above"

# ImageFolder needs only class folders in DATA_DIR; drop stray folders/files
for c in os.listdir(DATA_DIR):
    p = os.path.join(DATA_DIR, c)
    if c == '__MACOSX' or not os.path.isdir(p) or count_imgs(p) == 0:
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

print("\nDATA_DIR:", DATA_DIR)
for c in sorted(os.listdir(DATA_DIR)):
    print(c, count_imgs(os.path.join(DATA_DIR, c)), "images")

## 2. Load data

In [ ]:
import torch, torch.nn as nn, numpy as np
from torch.utils.data import DataLoader, Subset, RandomSampler
from torchvision import datasets, transforms, models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG, BATCH = 224, 64
torch.backends.cudnn.benchmark = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

full_train = datasets.ImageFolder(DATA_DIR, train_tf)
full_val = datasets.ImageFolder(DATA_DIR, val_tf)
class_names = full_train.classes            # alphabetical: ['infected', 'normal']
print("Classes:", class_names)

rng = np.random.RandomState(42)
idx = rng.permutation(len(full_train))
split = int(0.8 * len(idx))
# each epoch sees a random 2048-image sample (32 steps) instead of the full set -> much faster epochs
train_ds = Subset(full_train, idx[:split])
sampler = RandomSampler(train_ds, replacement=False, num_samples=min(2048, len(train_ds)))
train_dl = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4, pin_memory=True, persistent_workers=True, drop_last=True)
val_dl = DataLoader(Subset(full_val, idx[split:]), BATCH, num_workers=4, pin_memory=True, persistent_workers=True)
print(f"train {split} (using {len(train_dl)} steps/epoch) | val {len(idx) - split}")

## 3. Build model

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.last_channel, len(class_names)))
model = model.to(device)
criterion = nn.CrossEntropyLoss()

## 4. Train (head first, then light fine-tuning)

In [ ]:
import copy, time

scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))   # mixed precision = faster on GPU

def run_epoch(dl, opt=None):
    train = opt is not None
    model.train(train)
    loss_sum = correct = total = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                out = model(x)
                loss = criterion(out, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

best_acc, best_state = 0, None

def fit(epochs, lr, patience=25):
    global best_acc, best_state
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    bad = 0
    for ep in range(epochs):
        t = time.time()
        tl, ta = run_epoch(train_dl, opt)
        vl, va = run_epoch(val_dl)
        sched.step()
        print(f"epoch {ep+1}/{epochs} train acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time()-t:.0f}s")
        if va > best_acc:
            best_acc, best_state, bad = va, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break

# 5 + 20 = 25 epochs total
print("Stage 1: classifier head")
for p in model.features.parameters(): p.requires_grad = False
fit(epochs=5, lr=1e-3)

print("Stage 2: fine-tune whole network")
for p in model.parameters(): p.requires_grad = True
fit(epochs=20, lr=1e-4)

model.load_state_dict(best_state)
print("Best val accuracy: %.2f%%" % (100 * best_acc))

## 5. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        y_pred += model(x.to(device)).argmax(1).cpu().tolist()
        y_true += y.tolist()
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

## 6. Export to `models/` and download

In [ ]:
import json

class Deploy(nn.Module):
    # input: RGB image tensor (1,3,224,224) with raw 0-255 values -> output: class probabilities
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1) * 255)
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1) * 255)
    def forward(self, x):
        return torch.softmax(self.net((x - self.mean) / self.std), dim=1)

deploy = Deploy(model).to('cpu').eval()
scripted = torch.jit.trace(deploy, torch.rand(1, 3, IMG, IMG) * 255)

os.makedirs('models', exist_ok=True)
scripted.save('models/model.pt')
with open('models/labels.json', 'w') as f:
    json.dump(class_names, f)          # index order matches model output

print("Saved:", os.listdir('models'))
shutil.make_archive('models', 'zip', 'models')
files.download('models.zip')

## 7. Install on the Raspberry Pi
Unzip `models.zip` into the project's `models/` folder so you have `models/model.pt` and `models/labels.json`, then run `python app.py`.